In [2]:
# Setup: code comes from GitHub onto the VM disk; data + checkpoints live on Drive (see src/config.py)
import os, sys

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/s1156052/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN (never write it in the notebook)
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
!apt-get -qq install -y cd-hit mmseqs2 > /dev/null


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.
/content/s1156052/NeoAntigen


In [3]:
# Collect once (dated snapshot on Drive): every CEDAR neoantigen with a disease_group, plus viral/TAA background negatives
import pandas as pd
from src import config, pancancer

pd.set_option("display.width", 200)

raw = pancancer.get_raw(config.DATA_DIR, refresh=False)
print(raw.groupby("group").size())


using snapshot pancancer_cedar_raw_20261110.csv
group
neo      25011
taa      18161
viral    30488
dtype: int64


In [4]:
# How much data each disease has (decides which diseases can be their own train/test arm)
pancancer.summary(raw)


,disease_group,rows,unique_seq,positive_seq,exp_negative_seq,unknown_seq,median_year
0,melanoma,7106,7102,1534,5542,26,2021
1,lung,2453,2453,644,1792,17,2019
2,colorectal,8966,8957,462,8466,30,2019
3,pancreatic,1005,1005,235,768,2,2021
4,breast,483,483,220,250,13,2022
5,other,453,453,186,242,25,2024
6,bladder,183,183,172,11,0,2022
7,renal,242,242,142,100,0,2025
8,hepatocellular,560,560,138,420,2,2021
9,brain,527,527,133,393,1,2021


In [5]:
# Disease names that fell into "other": check none of them belongs to a named group or is not a cancer
pancancer.discover_terms(raw).head(40)


,rows
appendix carcinoma,135
gastrointestinal neuroendocrine tumor,89
squamous cell carcinoma,52
invasive ductal carcinoma,46
prostate cancer,19
medulloblastoma,15
collecting duct carcinoma,13
gastrointestinal system cancer,13
fibrolamellar carcinoma,11
salivary gland cancer,10


In [6]:
# Step 2: frozen dataset = one manifest (unique peptides, disease_group, CD-HIT cluster, outer fold). Needs cd-hit (Colab).
# neg_mode "exp": disease-matched T cell-negative neoantigens (primary) | "bg": viral/TAA background (the paper's choice)
from src import pancancer_split as ps
NAME, NEG_MODE = "pancancer_cedar_v1", "exp"
manifest, summary = ps.make_dataset(raw, config.DATA_DIR, config.WORK_DIR, name=NAME, neg_mode=NEG_MODE)
summary


,disease_group,role,N+,N-,prevalence,clusters,papers,paper_share
0,melanoma,core,1533,5531,0.217,2158,126,0.25
1,lung,core,644,1791,0.264,929,53,0.31
2,colorectal,core,461,8451,0.052,4243,50,0.62
3,pancreatic,ood,235,762,0.236,772,28,0.32
4,breast,ood,220,250,0.468,339,23,0.20
5,other,ood,184,242,0.432,332,49,0.62
6,bladder,ood,172,11,0.940,135,6,0.78
7,renal,ood,142,100,0.587,211,8,0.54
8,hepatocellular,ood,138,420,0.247,356,14,0.38
9,brain,ood,129,384,0.251,439,17,0.63


In [7]:
# Main layout (cluster split, no peptide thrown away): colorectal / lung / melanoma each ~80:20, pooled = their union,
# other cancers = extra test rows. One outer fold (FOLD) is the 20%; neg_ratio=None keeps every negative.
FOLD = 0
frames, sizes, evals = ps.write_main_arms(manifest, config.DATA_DIR, NAME, test_fold=FOLD, neg_ratio=None)
print(sizes.to_string(), "\n")      # train / own-test sizes per arm
evals[evals["arm"] == "pooled"]      # pooled test sets: N+ N- all rows vs "clean" rows (not similar to train); reportable needs >= 50


          arm  train N+  train N-  own test N+  own test N-
0  colorectal       384      6867           77         1584
1        lung       547      1480           97          311
2    melanoma      1265      4456          268         1075
3      pooled      2196     12803          442         2970 



,disease_group,in_dist,N+,N-,N+ clean,N- clean,reportable,arm
45,melanoma,True,268,1075,268,1075,True,pooled
46,lung,True,97,311,97,311,True,pooled
47,colorectal,True,77,1584,77,1584,True,pooled
48,pancreatic,False,235,762,44,279,False,pooled
49,breast,False,220,250,53,69,True,pooled
50,other,False,184,242,38,53,False,pooled
51,bladder,False,172,11,32,1,False,pooled
52,renal,False,142,100,27,29,False,pooled
53,hepatocellular,False,138,420,46,103,False,pooled
54,brain,False,129,384,27,143,False,pooled


In [ ]:
# Same dataset at 30% identity with MMseqs2 (CD-HIT cannot go below 0.4). New name, so the 0.4 version is kept.
NAME30 = "pancancer_cedar_v1_id30"
manifest30, summary30 = ps.make_dataset(raw, config.DATA_DIR, config.WORK_DIR, name=NAME30, sim_id=0.3, method="mmseqs")
frames30, sizes30, evals30 = ps.write_main_arms(manifest30, config.DATA_DIR, NAME30, test_fold=0, neg_ratio=None)
print(sizes30.to_string(), "\n")
print(ps.check_similarity(frames30["pooled"], sim_id=0.3).to_string())   # independent screen (ungapped, a lower bound)
evals30[evals30["arm"] == "pooled"]


In [8]:
# One arm (train diseases) x one outer test fold -> the dataframe the runner expects; written to DATA_DIR/<NAME>/<arm>_f<k>/
ARM, FOLD = "pooled", 0
df = ps.arm_frame(manifest, ARM, FOLD, neg_ratio=5)
ps.write_arm(df, config.DATA_DIR, NAME)
ps.eval_table(df)     # N+/N- per disease on test: all rows vs rows not similar to training (reportable needs >= 50 each)


,disease_group,in_dist,N+,N-,N+ clean,N- clean,reportable
0,melanoma,True,268,1075,268,1075,True
1,lung,True,97,311,97,311,True
2,colorectal,True,77,1584,77,1584,True
3,pancreatic,False,235,762,44,279,False
4,breast,False,220,250,53,69,True
5,other,False,184,242,38,53,False
6,bladder,False,172,11,32,1,False
7,renal,False,142,100,27,29,False
8,hepatocellular,False,138,420,46,103,False
9,brain,False,129,384,27,143,False


In [9]:
# Every arm x fold (cheap; each frame is checked for sequence/cluster leakage)
for arm in ps.ARMS:
    for k in range(5):
        ps.write_arm(ps.arm_frame(manifest, arm, k), config.DATA_DIR, NAME)


In [10]:
# OPTIONAL comparison with the paper (pooled model only): raw -> pos/neg -> CD-HIT 0.4 representatives -> length <= 35 -> 80:20.
# CD-HIT keeps one peptide per cluster, so positives collapse (colorectal 462 -> 63): the per-cancer tests become too small.
# colorectal / lung / melanoma each 80:20; pooled = the three together; other cancers = extra test only
from src import pancancer_dataset as pds
NAME_P = "pancancer_paper_v1"
res = pds.make_paper_dataset(raw, config.DATA_DIR, config.WORK_DIR, name=NAME_P)
print(res["table1"].to_string(index=False), "\n")   # per cancer: raw -> CD-HIT -> length, and how much survives
print(res["table2"].to_string(), "\n")      # train / test sizes per set
print(res["similarity"].to_string(), "\n")  # max identity of each held-out set to the pooled train (violations should be 0)
print("limitations:", res["limitations"])


            cancer                    raw    after CD-HIT 0.4    after length<=35 positives kept negatives kept
          melanoma  1,534 pos / 5,540 neg 136 pos / 1,673 neg 135 pos / 1,665 neg             9%            30%
        colorectal    462 pos / 8,466 neg  63 pos / 2,078 neg  62 pos / 2,063 neg            13%            24%
              lung    644 pos / 1,792 neg    34 pos / 207 neg    34 pos / 206 neg             5%            11%
        pancreatic      235 pos / 768 neg    31 pos / 224 neg    31 pos / 218 neg            13%            28%
            breast      220 pos / 250 neg     40 pos / 23 neg     40 pos / 23 neg            18%             9%
      other (misc)      185 pos / 242 neg      13 pos / 6 neg      12 pos / 6 neg             6%             2%
           bladder       172 pos / 11 neg       0 pos / 1 neg       0 pos / 1 neg             0%             9%
             renal      142 pos / 100 neg      16 pos / 7 neg      16 pos / 7 neg            11%        

In [11]:
# Load one set for the runner: data.load_dataset(config.DATA_DIR / NAME_P / "colorectal" | "lung" | "melanoma" | "pooled")
from src import data
df_pool = data.load_dataset(config.DATA_DIR / NAME_P / "pooled")
pds.eval_table(df_pool)     # test sets of the pooled model: 3 cancers + each other cancer + all other solid


,test_set,N+,N-,reportable
0,other_solid (all),188,1036,True
1,other_head_neck,42,266,False
2,other_breast,40,23,False
3,other_pancreatic,31,218,False
4,other_hepatocellular,30,33,False
5,melanoma,27,333,False
6,other_renal,16,7,False
7,hematologic,14,6,False
8,colorectal,12,413,False
9,other_misc,12,6,False
